# 🚚 Delivery Dataset — Cleaning & Feature Engineering
---
**Dataset:** `delivery.csv`  
**Goal:** Clean delivery records, validate key fields, and flag high-risk or problematic deliveries.  
**Final output:** `delivery_cleaned.csv`

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

## 2. Load Dataset

In [2]:
# Load the raw delivery CSV file
df = pd.read_csv("delivery.csv")

# Quick shape check
print("Shape:", df.shape)
df.head()

Shape: (20000, 9)


,delivery_id,order_id,delivery_partner,delivery_status,delivery_time_minutes,distance_km,traffic_level,weather_condition,late_delivery
0,1,1,SwiftEat,Delivered,38,10.1,Medium,Clear,No
1,2,2,QuickDrop,Cancelled,84,10.9,High,Rainy,Yes
2,3,3,QuickDrop,Delivered,40,13.4,High,Clear,Yes
3,4,4,SwiftEat,Delivered,24,3.3,Medium,Clear,No
4,5,5,SwiftEat,Cancelled,45,14.0,Medium,Clear,No


## 3. Explore the Data

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   delivery_id            20000 non-null  int64  
 1   order_id               20000 non-null  int64  
 2   delivery_partner       20000 non-null  str    
 3   delivery_status        20000 non-null  str    
 4   delivery_time_minutes  20000 non-null  int64  
 5   distance_km            20000 non-null  float64
 6   traffic_level          20000 non-null  str    
 7   weather_condition      20000 non-null  str    
 8   late_delivery          20000 non-null  str    
dtypes: float64(1), int64(3), str(5)
memory usage: 1.4 MB


In [5]:
df.describe()

,delivery_id,order_id,delivery_time_minutes,distance_km
count,20000.000000,20000.000000,20000.000000,20000.000000
mean,10000.500000,10000.500000,43.443450,8.038385
std,5773.647028,5773.647028,13.641701,4.048607
min,1.000000,1.000000,20.000000,1.000000
25%,5000.750000,5000.750000,33.000000,4.500000
50%,10000.500000,10000.500000,43.000000,8.100000
75%,15000.250000,15000.250000,52.000000,11.500000
max,20000.000000,20000.000000,94.000000,15.000000


## 4. Check Duplicates & Missing Values

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df.isnull().sum()

delivery_id              0
order_id                 0
delivery_partner         0
delivery_status          0
delivery_time_minutes    0
distance_km              0
traffic_level            0
weather_condition        0
late_delivery            0
dtype: int64

## 5. Data Validation

In [4]:
# Delivery time must be a positive number (0 or negative makes no sense)
invalid_time = df[df['delivery_time_minutes'] <= 0]
print(f"Invalid delivery time rows: {len(invalid_time)}")

Invalid delivery time rows: 0


In [5]:
# Distance must be positive
invalid_dist = df[df['distance_km'] <= 0]
print(f"Invalid distance rows: {len(invalid_dist)}")

Invalid distance rows: 0


In [6]:
# --- Categorical Columns — check for unexpected values ---
print("\ndelivery_status unique values:", df['delivery_status'].unique())
print("traffic_level unique values   :", df['traffic_level'].unique())
print("weather_condition unique values:", df['weather_condition'].unique())


delivery_status unique values: <StringArray>
['Delivered', 'Cancelled', 'Refunded']
Length: 3, dtype: str
traffic_level unique values   : <StringArray>
['Medium', 'High', 'Low']
Length: 3, dtype: str
weather_condition unique values: <StringArray>
['Clear', 'Rainy', 'Fog', 'Storm']
Length: 4, dtype: str


## 6. Feature Engineering

In [8]:
# Bin deliveries into Fast / Moderate / Slow based on time taken
df['delivery_speed'] = pd.cut(
df['delivery_time_minutes'],
bins=[0,30,60,200],
labels=['Fast','Moderate','Slow'])

In [13]:
# Orders beyond 10 km are classified as long distance
df['long_distance_order'] = np.where( df['distance_km'] > 10,'Yes', 'No')

In [14]:
# Flag orders that occurred during high traffic conditions
df['high_traffic'] = np.where(df['traffic_level'] == 'High','Yes','No')

In [15]:
# Rainy and Storm conditions are considered bad weather for delivery
df['bad_weather'] = np.where(
    df['weather_condition'].isin(['Rainy', 'Storm']),
    'Yes', 'No' )

## 7. Final Check & Export

In [9]:
# Final shape and null check before saving
print("Final Shape:", df.shape)
print("\nNull values after all cleaning steps:")
print(df.isnull().sum())
df.head()

Final Shape: (20000, 10)

Null values after all cleaning steps:
delivery_id              0
order_id                 0
delivery_partner         0
delivery_status          0
delivery_time_minutes    0
distance_km              0
traffic_level            0
weather_condition        0
late_delivery            0
delivery_speed           0
dtype: int64


,delivery_id,order_id,delivery_partner,delivery_status,delivery_time_minutes,distance_km,traffic_level,weather_condition,late_delivery,delivery_speed
0,1,1,SwiftEat,Delivered,38,10.1,Medium,Clear,No,Moderate
1,2,2,QuickDrop,Cancelled,84,10.9,High,Rainy,Yes,Slow
2,3,3,QuickDrop,Delivered,40,13.4,High,Clear,Yes,Moderate
3,4,4,SwiftEat,Delivered,24,3.3,Medium,Clear,No,Fast
4,5,5,SwiftEat,Cancelled,45,14.0,Medium,Clear,No,Moderate


In [10]:
df.to_csv("delivery_cleaned.csv", index=False)
print("✅ delivery_cleaned.csv saved successfully!")

✅ delivery_cleaned.csv saved successfully!
